In [1]:
# --- 1. LIBRARIES ---
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning & Scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_curve, auc)

# Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (Dense, Conv1D, BatchNormalization, 
                                     concatenate, Flatten, Dropout, Attention, 
                                     Reshape, GlobalAveragePooling1D, Multiply, MaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:

# --- 2. REPRODUCIBILITY (The "Same Result" Rule) ---
def set_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)
    print(f"Environment seeds locked at {seed}")

set_seeds(42)

Environment seeds locked at 42


In [3]:
# Load the specific binary dataset
df = pd.read_csv("dataset_name.csv", engine="pyarrow")

# Verify the load by checking the total rows and columns
print(f"Dataset successfully loaded.")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

Dataset successfully loaded.
Total Rows: 100309665
Total Columns: 84


In [4]:
# 1. Strip any leading/trailing whitespace and convert to lowercase
df.columns = df.columns.str.strip().str.lower()

# 2. Replace spaces, slashes, and dots with underscores for easy coding
df.columns = df.columns.str.replace(' ', '_', regex=False)
df.columns = df.columns.str.replace('/', '_', regex=False)
df.columns = df.columns.str.replace('.', '_', regex=False)

# 3. Print the first 10 columns to verify the change
print("First 10 standardized columns:")
print(df.columns[:10].tolist())

# 4. Find the exact name of your Label column
label_col = [col for col in df.columns if 'label' in col]
print(f"\nLabel column found: {label_col}")

First 10 standardized columns:
['flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts']

Label column found: ['label']


In [5]:
# List of columns that are not useful for training features
identifiers = ['flow_id', 'src_ip', 'dst_ip', 'timestamp']

# Drop them only if they exist in the current dataframe
df.drop(columns=[col for col in identifiers if col in df.columns], inplace=True)

print(f"Identifiers removed.")
print(f"Remaining columns: {df.shape[1]}")

Identifiers removed.
Remaining columns: 80


In [6]:
df.shape

(100309665, 80)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100309665 entries, 0 to 100309664
Data columns (total 80 columns):
 #   Column             Dtype  
---  ------             -----  
 0   src_port           int64  
 1   dst_port           int64  
 2   protocol           int64  
 3   flow_duration      int64  
 4   tot_fwd_pkts       int64  
 5   tot_bwd_pkts       int64  
 6   totlen_fwd_pkts    float64
 7   totlen_bwd_pkts    float64
 8   fwd_pkt_len_max    float64
 9   fwd_pkt_len_min    float64
 10  fwd_pkt_len_mean   float64
 11  fwd_pkt_len_std    float64
 12  bwd_pkt_len_max    float64
 13  bwd_pkt_len_min    float64
 14  bwd_pkt_len_mean   float64
 15  bwd_pkt_len_std    float64
 16  flow_byts_s        float64
 17  flow_pkts_s        float64
 18  flow_iat_mean      float64
 19  flow_iat_std       float64
 20  flow_iat_max       float64
 21  flow_iat_min       float64
 22  fwd_iat_tot        float64
 23  fwd_iat_mean       float64
 24  fwd_iat_std        float64
 25  fwd_iat_max   

In [8]:
df.drop_duplicates(inplace=True)

In [9]:
df['label'].value_counts()

label
1    90694000
0     7331139
Name: count, dtype: int64

In [10]:
import joblib
import numpy as np

imputer_bundle = joblib.load("regression_imputer.pkl")

imputer_models = imputer_bundle['models']
COLS_WONA = imputer_bundle['cols_wona']
COLS_WNA = imputer_bundle['cols_wna']


In [11]:
# Replace Inf with NaN (same as training)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [12]:
print("Remaining NaNs:", df.isna().sum().sum())

Remaining NaNs: 2788764


In [13]:
import gc

chunk_size = 5_000_000

for target_col, model in imputer_models.items():
    if target_col not in df.columns:
        continue  # safety check

    # Get indices of missing values
    predict_idx = df.index[df[target_col].isna()]
    
    if len(predict_idx) > 0:
        print(f"Applying predictions for: {target_col}")
        
        # Predict and fill in chunks
        for i in range(0, len(predict_idx), chunk_size):
            idx_chunk = predict_idx[i : i + chunk_size]
            X_pred_chunk = df.loc[idx_chunk, COLS_WONA].values
            
            preds = model.predict(X_pred_chunk)
            df.loc[idx_chunk, target_col] = preds
            
            del X_pred_chunk, preds
            
    # Force RAM cleanup after each column
    gc.collect()

Applying predictions for: flow_byts_s


C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


Applying predictions for: flow_pkts_s


C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [14]:
print("Remaining NaNs:", df.isna().sum().sum())


Remaining NaNs: 0


In [15]:
# 1. Check for any columns that are not integers or floats (excluding 'label')
non_numeric = df.drop(columns=['label']).select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Non-numeric columns found (excluding label): {non_numeric}")

# 2. Check for Infinite values (common in this dataset)
inf_count = np.isinf(df.select_dtypes(include=[np.number])).values.sum()
print(f"Total Infinite values: {inf_count}")

# 3. Check for Null/NaN values
nan_count = df.isnull().sum().sum()
print(f"Total NaN values: {nan_count}")

Non-numeric columns found (excluding label): []
Total Infinite values: 0
Total NaN values: 0


In [16]:
# 1. Check unique values in the label column
print("Unique labels in dataset:")
print(df['label'].unique())

# 2. Check the count of each label to see if it is imbalanced
print("\nLabel counts:")
print(df['label'].value_counts())

# 3. Check the data type of the label
print(f"\nLabel data type: {df['label'].dtype}")

Unique labels in dataset:
[1 0]

Label counts:
label
1    90694000
0     7331139
Name: count, dtype: int64

Label data type: int64


In [17]:
import gc
import joblib

# 1. Define Features (X) and Target (y)
X = df.drop(columns=['label'])
y = df['label']

# CLEAR RAM: Original dataframe is no longer needed
del df
gc.collect()

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# CLEAR RAM: X is no longer needed
del X
gc.collect()

print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set:  {X_test.shape[0]} rows")

scaler = joblib.load("scaler.pkl")
print("Scaler loaded successfully")

X_train_scaled = scaler.transform(X_train)
# CLEAR RAM: Delete unscaled X_train
del X_train
gc.collect()

X_test_scaled = scaler.transform(X_test)
# CLEAR RAM: Delete unscaled X_test
del X_test
gc.collect()

print(f"Scaling Complete. RAM cleaned.")
print(f"Train dtype: {X_train_scaled.dtype}")
print(f"Train shape: {X_train_scaled.shape}")

Training set: 78420111 rows
Testing set:  19605028 rows
Scaler loaded successfully
Scaling Complete. RAM cleaned.
Train dtype: float64
Train shape: (78420111, 79)


In [ ]:
import pandas as pd
import numpy as np
import gc
import itertools
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.utils import resample
import joblib

# 1. --- CRITICAL MEMORY OPTIMIZATION ---
print("Ensuring float32 memory format...")
X_train_scaled = X_train_scaled.astype(np.float32)
X_test_scaled = X_test_scaled.astype(np.float32)
gc.collect()

# 2. --- CREATE THE TUNING SUBSET ---
# We use 1,000,000 rows for tuning to finish in hours rather than weeks.
# 'stratify=y_train' ensures the benign/attack ratio remains perfectly balanced.
print("\nExtracting 1,000,000 rows for rapid hyperparameter tuning...")
X_tune, y_tune = resample(X_train_scaled, y_train, 
                          n_samples=1000000, 
                          replace=False, 
                          stratify=y_train, 
                          random_state=42)

# 3. --- DEFINE THE GRID ---
depths = [5, 10, 15, 20]
splits = [50, 100, 250]
leaves = [15, 50, 100]

combinations = list(itertools.product(depths, splits, leaves))
print(f"Total combinations to test: {len(combinations)}")

# 4. --- INITIALIZE TRACKING VARIABLES ---
best_test_acc = 0.0
best_params = None
overfit_tolerance = 0.015 # Maximum allowed gap between Train and Test (1.5%)

results_log = []

print("\n--- STARTING HYPERPARAMETER GRID SEARCH ---")
print(f"{'Depth':<7} | {'Split':<7} | {'Leaf':<7} | {'Train Acc':<10} | {'Test Acc':<10} | {'Gap':<8} | {'Status'}")
print("-" * 75)

# 5. --- THE SEARCH LOOP ---
for depth, split, leaf in combinations:
    
    # Initialize model for this combination
    rf_temp = RandomForestClassifier(
        n_estimators=50,          # Reduced to 50 just for the tuning phase to speed it up
        max_depth=depth,
        min_samples_split=split,
        min_samples_leaf=leaf,
        class_weight='balanced',
        n_jobs=8,                 # Safe for 128GB RAM
        random_state=42
    )
    
    # Train on the subset
    rf_temp.fit(X_tune, y_tune)
    
    # Predict on subset train AND the massive test set
    train_preds = rf_temp.predict(X_tune)
    test_preds = rf_temp.predict(X_test_scaled) # Always test against the full test set
    
    # Calculate Accuracies
    train_acc = accuracy_score(y_tune, train_preds)
    test_acc = accuracy_score(y_test, test_preds)
    gap = train_acc - test_acc
    
    # Evaluate Overfitting
    if gap > overfit_tolerance:
        status = "OVERFIT ❌"
    elif gap < 0:
        status = "UNDERFIT ⚠️"
    else:
        status = "VALID ✅"
        # Only update "best" if the model is valid (not overfit)
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_params = {'depth': depth, 'split': split, 'leaf': leaf}
            
    # Log results for visualization later
    results_log.append({
        'Depth': depth, 'Split': split, 'Leaf': leaf, 
        'Train_Acc': train_acc, 'Test_Acc': test_acc, 'Gap': gap, 'Status': status
    })
    
    # Print real-time progress
    print(f"{depth:<7} | {split:<7} | {leaf:<7} | {train_acc:.5f}  | {test_acc:.5f}  | {gap:+.5f} | {status}")
    
    # Free RAM after every loop
    del rf_temp, train_preds, test_preds
    gc.collect()

print("-" * 75)
print("\n--- TUNING COMPLETE ---")

# Convert log to dataframe so you can export/view it easily
df_results = pd.DataFrame(results_log)
df_results.to_csv("Hyperparameter_Tuning_Results.csv", index=False)

if best_params is None:
    print("WARNING: All models overfit based on the strict 1.5% tolerance.")
    print("Consider lowering max_depth further or increasing min_samples_leaf.")
else:
    print("\n🏆 WINNING COMBINATION (Zero Overfitting):")
    print(f"Max Depth:         {best_params['depth']}")
    print(f"Min Samples Split: {best_params['split']}")
    print(f"Min Samples Leaf:  {best_params['leaf']}")
    print(f"Validation Acc:    {best_test_acc:.5f}")
    
    print("\n--- TRAINING FINAL FULL MODEL ---")
    # Now we train 100 trees using the exact best params on ALL data
    final_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=best_params['depth'],
        min_samples_split=best_params['split'],
        min_samples_leaf=best_params['leaf'],
        class_weight='balanced',
        max_samples=0.8,        # Add subsampling to the final model for extra safety
        n_jobs=8,
        random_state=42,
        verbose=1
    )
    
    # Train on the FULL dataset
    print(f"Training final model on {X_train_scaled.shape[0]} rows. This will take some time...")
    final_rf.fit(X_train_scaled, y_train)
    
    joblib.dump(final_rf, "Final_Tuned_RF_Classifier.pkl")
    print("Final tuned model saved successfully!")

In [ ]:
# --- ADVANCED EVALUATION VISUALIZATIONS ---
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import warnings
warnings.filterwarnings("ignore")

# Extract the probability of the 'Attack' class
y_probs_attack = test_probs[:, 1]

# 1. Setup the figure for the first 3 fast plots
plt.figure(figsize=(18, 5))

# --- PLOT 1: Precision-Recall (PR) Curve ---
plt.subplot(1, 3, 1)
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_probs_attack)
avg_precision = average_precision_score(y_test, y_probs_attack)

plt.plot(recall, precision, color='purple', lw=2, label=f'PR Curve (AP = {avg_precision:.4f})')
plt.xlabel('Recall (True Positive Rate)', fontsize=12)
plt.ylabel('Precision (Positive Predictive Value)', fontsize=12)
plt.title('Precision-Recall Curve', fontsize=14)
plt.legend(loc="lower left")
plt.grid(alpha=0.3)

# --- PLOT 2: F1-Score vs. Decision Threshold ---
plt.subplot(1, 3, 2)
# F1 = 2 * (Precision * Recall) / (Precision + Recall)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10) 
best_threshold = thresholds_pr[np.argmax(f1_scores)]

plt.plot(thresholds_pr, f1_scores[:-1], color='teal', lw=2, label='F1 Score')
plt.axvline(x=best_threshold, color='red', linestyle='--', label=f'Best Threshold ({best_threshold:.2f})')
plt.xlabel('Decision Threshold', fontsize=12)
plt.ylabel('F1 Score', fontsize=12)
plt.title('Optimal Threshold Selection', fontsize=14)
plt.legend(loc="lower center")
plt.grid(alpha=0.3)

# --- PLOT 3: Calibration Curve (Reliability Diagram) ---
plt.subplot(1, 3, 3)
prob_true, prob_pred = calibration_curve(y_test, y_probs_attack, n_bins=10)

plt.plot(prob_pred, prob_true, marker='o', color='darkorange', lw=2, label='RF Calibration')
plt.plot([0, 1], [0, 1], linestyle='--', color='navy', label='Perfectly Calibrated')
plt.xlabel('Mean Predicted Probability', fontsize=12)
plt.ylabel('Fraction of Actual Attacks', fontsize=12)
plt.title('Calibration Curve', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- PLOT 4: SHAP Values (Explainable AI) ---
print("\nCalculating SHAP values for Explainable AI (This may take 1-3 minutes)...")

# CRITICAL TIME-SAVER: We only sample 1,000 random rows from the test set. 
# Attempting SHAP on the whole test set will crash the server.
sample_indices = np.random.choice(X_test_scaled.shape[0], size=1000, replace=False)
X_test_sample = X_test_scaled[sample_indices]

# Attempt to get actual feature names, or fall back to generic names
try:
    feature_names = scaler.feature_names_in_
except AttributeError:
    feature_names = [f"Feature {i}" for i in range(X_test_scaled.shape[1])]

# Convert sample to DataFrame so SHAP can automatically use the column names in the plot
df_sample = pd.DataFrame(X_test_sample, columns=feature_names)

# Initialize the Tree Explainer
explainer = shap.TreeExplainer(final_rf_model)
shap_values = explainer.shap_values(df_sample)

# For a binary Random Forest, shap_values is a list of arrays: [Class 0, Class 1]. 
# We want to visualize what drives Class 1 (the Attack class).
plt.figure(figsize=(10, 8))
plt.title("SHAP Summary Plot: Features driving 'Attack' predictions", fontsize=14, pad=20)
shap.summary_plot(shap_values[1], df_sample, show=False) # show=False allows us to size the figure
plt.tight_layout()
plt.show()

print("All advanced visualizations successfully generated.")